In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Subject–Frequency–Channel ICA on Wavelet Power

## Scope

This notebook prepares the data and runs the ICA decomposition on the
`(T, F × C × S)` reshape of the 4-D wavelet power tensor. Time forms
the observation axis; subjects, channels and frequencies are combined
into the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_times,  n_freqs × n_channels × n_subjects)
         ── obs ──  ─────────── features ──────────
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.  This removes overall amplitude differences and ensures that
PCA/ICA operates on standardised activations.

ICA components live in the `F × C × S` feature space, so each
component is reshaped back to `(n_freqs, n_channels, n_subjects)` — a
**frequency × channel × subject pattern** that is jointly active in
time. The corresponding ICA scores are 1-D **temporal activations**
indicating when each pattern is expressed across the recording.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time and reshape to `(T, F×C×S)`.
4. PCA dimensionality reduction.
5. FastICA on the PCA scores.

After the final cell the following variables are available for any
downstream analysis:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_t` | `(T, F×C×S)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(T, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(T, K_ica)` | ICA scores (temporal activation per IC) |
| `ica_components` | `(K_ica, F×C×S)` | ICA freq–channel–subject patterns |
| `components_3d` | `(K, F, C, S)` | ICA components reshaped to freq × chan × subj |

## Configuration

In [ ]:
# ── Experiment configuration ────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ─────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ───────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ──────────────────────────────────────────────
USE_PCA = True  # set False to run FastICA directly on the (T, F*C*S) matrix
N_COMPONENTS_PCA = 50  # number of PCA components to retain (ignored when USE_PCA=False)
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "subject_frequency_channel"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Use PCA                : {USE_PCA}")
if USE_PCA:
    print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.  This ensures that PCA/ICA are not
dominated by high-power channels, subjects, or frequency bands.

**Reshaping** moves time to the observation axis and collapses
frequencies, channels and subjects into a single feature axis:

```
(S, C, F, T)  →  transpose to  (T, F, C, S)
              →  reshape to     (T,  F × C × S)
                                obs   features
```

Each row of the resulting 2-D matrix is the z-scored power, at a single
time point, for every `(frequency, channel, subject)` combination.
PCA/ICA will discover **freq–channel–subject patterns** — joint
spectro-spatial-individual fingerprints — whose temporal activation
varies through the recording.

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (T, F, C, S) → (T, F*C*S)
bb_z_t = bb_z.transpose(3, 2, 1, 0)  # (T, F, C, S)
n_obs = n_times
n_feat = n_freqs * n_channels * n_subjects
X_t = bb_z_t.reshape(n_obs, n_feat)  # (T, F*C*S)

print(f"Reshaped matrix shape : {X_t.shape}")
print(f"  Observations (T)    : {X_t.shape[0]}")
print(f"  Features (F×C×S)    : {X_t.shape[1]}")
print(f"Column means ≈ 0 : {X_t.mean(axis=0).mean():.6f}")
print(f"Column stds        : {X_t.std(axis=0).mean():.4f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the `F × C × S` feature space to `N_COMPONENTS_PCA`
principal components, keeping the directions of maximum variance.  Then
FastICA rotates the PCA subspace to maximise statistical independence,
yielding `N_COMPONENTS_ICA` independent components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(T, K)` | Temporal activation per IC |
| `ica_components` | `(K, F×C×S)` | Freq–channel–subject pattern per IC |
| `components_3d` | `(K, F, C, S)` | ICA components reshaped to freq × chan × subj |

In [ ]:
# --- PCA (optional) ---
if USE_PCA:
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
    pca_scores = pca.fit_transform(X_t)  # (T, K_pca)

    explained = pca.explained_variance_ratio_
    cumulative = np.cumsum(explained)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
    axes[0].set_xlabel("Component")
    axes[0].set_ylabel("Variance explained")
    axes[0].set_title(f"PCA Scree Plot — {LABEL}")

    axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
    axes[1].axhline(0.9, ls="--", color="gray", label="90%")
    axes[1].set_xlabel("Number of components")
    axes[1].set_ylabel("Cumulative variance explained")
    axes[1].set_title(f"Cumulative Variance — {LABEL}")
    axes[1].legend()

    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")

    print(
        f"Top {N_COMPONENTS_PCA} components explain "
        f"{cumulative[-1] * 100:.1f}% of total variance."
    )
else:
    print("PCA skipped — FastICA will be fit directly on X_t.")

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
if USE_PCA:
    ica_scores = ica.fit_transform(pca_scores)  # (T, K_ica)
    ica_components = ica.components_ @ pca.components_  # (K_ica, F*C*S)
else:
    ica_scores = ica.fit_transform(X_t)  # (T, K_ica)
    ica_components = ica.components_  # (K_ica, F*C*S)

# Reshape ICA components to (K, F, C, S) for downstream analysis
components_3d = ica_components.reshape(
    N_COMPONENTS_ICA, n_freqs, n_channels, n_subjects
)  # (K, F, C, S)

print(f"ICA scores shape       : {ica_scores.shape}  (T, K)")
print(f"ICA components shape   : {ica_components.shape}  (K, F*C*S)")
print(f"Components 3-D shape   : {components_3d.shape}  (K, F, C, S)")

---
## Analysis (a) — Intersubject Correlation Matrix

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
**(frequency × channel)** loading map for that component, flattened
to a vector of length `F × C`:

```
subj_maps[k, s, :] = components_3d[k, :, :, s].reshape(F*C)
corr_mat[k]        = corrcoef( subj_maps[k] )    # (S, S)
```

High off-diagonal correlations indicate that the component captures a
consistent spectro-spatial pattern across individuals — a hallmark of
stimulus-driven rather than noise-driven modes.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

im = None
for i, ax in enumerate(axes):
    # components_3d[i] has shape (F, C, S); transpose so subject is first,
    # then flatten F×C → one row per subject of length F*C.
    subj_maps = components_3d[i].transpose(2, 0, 1).reshape(n_subjects, -1)
    corr_mat = np.corrcoef(subj_maps)  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Loading Maps (F×C) — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Time–Frequency Map per Component (Outer Product)

For each ICA component we build a **frequency × time** map directly
from the ICA decomposition:

```
freq_profile[k]  = mean_(c, s) [ components_3d[k] ]   # (K, F)
time_profile[k]  = ica_scores[:, k]                    # (K, T)
tf_map[k]        = outer(freq_profile[k], time_profile[k])  # (K, F, T)
```

The outer product gives a rank-1 approximation of the component's
frequency–time structure grounded entirely in the ICA solution. A
diverging colormap (`RdBu_r`, symmetric around zero) preserves the
sign of the values, with a per-IC limit so weak components are not
flattened by stronger ones.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `freq_profiles` | `(K, F)` | Channel- and subject-averaged loading per frequency |
| `time_profiles` | `(K, T)` | ICA temporal activation per IC |
| `ft_maps` | `(K, F, T)` | Outer-product frequency × time map per component |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# Frequency profile: collapse channels and subjects  (K, F, C, S) → (K, F)
freq_profiles = components_3d.mean(axis=(2, 3))  # (K, F)

# Time profile: ICA temporal activation  (T, K) → (K, T)
time_profiles = ica_scores.T  # (K, T)

# Outer product per component: (K, F) x (K, T) → (K, F, T)
ft_maps = np.einsum("kf,kt->kft", freq_profiles, time_profiles)  # (K, F, T)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_maps[i]  # (F, T)
    vlim_i = np.percentile(np.abs(data_i), 99)
    mesh = ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="RdBu_r",
        vmin=-vlim_i,
        vmax=vlim_i,
        shading="auto",
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} — Freq × Time Map (outer product)", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency × Time Maps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Mean LOO-ISC Across Participants per Component

Single-scalar summary of Analysis (a): for each ICA component the
per-subject vector is the subject's **flattened (frequency × channel)
loading map** (`components_3d[k, :, :, s]` reshaped to `F·C`). The
leave-one-out inter-subject correlation is then:

```
for each subject s:
    subj_vec  = components_3d[k, :, :, s].reshape(-1)        # (F*C,)
    others    = mean over s' ≠ s  of  components_3d[k, :, :, s'].reshape(-1)
    r[k, s]   = pearson( subj_vec, others )
```

The bar height is the **mean across subjects** of `r[k, s]`; the error
bar is the across-subject std. High bars mark components whose
freq–channel loading map is reproduced consistently across
participants.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_per_subject` | `(K, S)` | LOO-ISC of (freq × chan) loading maps per IC and subject |
| `loo_isc_mean` | `(K,)` | Across-subject mean LOO-ISC per IC |
| `loo_isc_std` | `(K,)` | Across-subject std LOO-ISC per IC |

In [ ]:
from scipy.stats import pearsonr  # noqa: E402

# Per-subject vector for each IC is the subject's flattened (freq × chan)
# loading map of length F*C; matches the subject vectors used in Analysis (a).
loo_isc_per_subject = np.zeros((N_COMPONENTS_ICA, n_subjects))
for k in range(N_COMPONENTS_ICA):
    # components_3d[k] has shape (F, C, S); transpose so subject is first
    subj_vectors = components_3d[k].transpose(2, 0, 1).reshape(n_subjects, -1)
    for s in range(n_subjects):
        others_mean = np.delete(subj_vectors, s, axis=0).mean(axis=0)
        loo_isc_per_subject[k, s] = float(pearsonr(subj_vectors[s], others_mean)[0])

loo_isc_mean = loo_isc_per_subject.mean(axis=1)  # (K,)
loo_isc_std = loo_isc_per_subject.std(axis=1)  # (K,)

# Red bars where the across-subject mean LOO-ISC is negative, blue otherwise.
bar_colors = ["firebrick" if m < 0 else "steelblue" for m in loo_isc_mean]

fig, ax = plt.subplots(figsize=(max(8, 0.9 * N_COMPONENTS_ICA), 4.5))
xs = np.arange(N_COMPONENTS_ICA)
ax.bar(
    xs,
    loo_isc_mean,
    yerr=loo_isc_std,
    color=bar_colors,
    capsize=4,
)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels([f"IC {k + 1}" for k in range(N_COMPONENTS_ICA)])
ax.set_xlabel("Component")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"Per-IC Mean LOO-ISC Across Participants — {LABEL}")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_loo_isc_bar.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (d) — Pairwise Component Heatmaps

Each ICA component lives in the 3-D feature space `(F, C, S)`. To
inspect it as a 2-D heatmap we collapse one axis by **averaging** over
it, yielding the three pairwise views:

```
subject × frequency  : components_3d.mean(axis=channels)   → (K, F, S)
subject × channel    : components_3d.mean(axis=freqs)      → (K, C, S)
frequency × channel  : components_3d.mean(axis=subjects)   → (K, F, C)
```

The figure below has one column per component and three rows, one per
pairwise view. Color is symmetric around zero (`RdBu_r`) with a
per-panel 99-percentile limit so weak components stay legible.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `sf_maps` | `(K, F, S)` | Subject × frequency loading (mean over channels) |
| `sc_maps` | `(K, C, S)` | Subject × channel loading (mean over frequencies) |
| `fc_maps` | `(K, F, C)` | Frequency × channel loading (mean over subjects) |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# components_3d shape: (K, F, C, S)
sf_maps = components_3d.mean(axis=2)  # (K, F, S) — mean over channels
sc_maps = components_3d.mean(axis=1)  # (K, C, S) — mean over frequencies
fc_maps = components_3d.mean(axis=3)  # (K, F, C) — mean over subjects

subject_ticks = np.arange(1, n_subjects + 1)

pairwise_views = [
    {
        "maps": sf_maps,
        "title": f"Subject × Frequency (mean over channels) — {LABEL}",
        "xlabel": "Subject",
        "ylabel": "Frequency (Hz)",
        "extent": [0.5, n_subjects + 0.5, FREQS[0], FREQS[-1]],
        "xticks": subject_ticks,
        "filename": "ica_pairwise_subject_frequency.png",
    },
    {
        "maps": sc_maps,
        "title": f"Subject × Channel (mean over frequencies) — {LABEL}",
        "xlabel": "Subject",
        "ylabel": "Channel",
        "extent": [0.5, n_subjects + 0.5, 0.5, n_channels + 0.5],
        "xticks": subject_ticks,
        "filename": "ica_pairwise_subject_channel.png",
    },
    {
        "maps": fc_maps,
        "title": f"Frequency × Channel (mean over subjects) — {LABEL}",
        "xlabel": "Channel",
        "ylabel": "Frequency (Hz)",
        "extent": [0.5, n_channels + 0.5, FREQS[0], FREQS[-1]],
        "xticks": None,
        "filename": "ica_pairwise_frequency_channel.png",
    },
]

for view in pairwise_views:
    fig, axes = plt.subplots(
        1, n_show, figsize=(3.2 * n_show, 3.6), constrained_layout=True
    )
    if n_show == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        data_i = view["maps"][i]
        lim_i = np.percentile(np.abs(data_i), 99) or 1e-12
        im = ax.imshow(
            data_i,
            aspect="auto",
            cmap="RdBu_r",
            vmin=-lim_i,
            vmax=lim_i,
            origin="lower",
            extent=view["extent"],
        )
        if view["xticks"] is not None:
            ax.set_xticks(view["xticks"])
        ax.set_xlabel(view["xlabel"])
        ax.set_title(f"IC {i + 1}", fontsize=10)
        if i == 0:
            ax.set_ylabel(view["ylabel"])
        fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

    fig.suptitle(view["title"], fontsize=13)

    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / view["filename"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")